In [7]:
#This script takes two .txt Bible files and the plg_verses.csv as input and returns aligned ENG+LANG files with all matching verse numbers

In [7]:
from pathlib import Path

def load_bible_txt_as_dict(path: str | Path) -> dict[str, str]:
    """
    Parse a monolingual Bible .txt file:
    - Skips metadata lines starting with '#'
    - Expects: '<VERSE_ID><whitespace><TEXT...>'
    Returns a dict mapping verse_id -> text.
    """
    path = Path(path)
    verses: dict[str, str] = {}

    with path.open("r", encoding="utf-8") as f:
        for raw in f:
            line = raw.strip()
            if not line or line.startswith("#"):
                continue
            # Split once: supports tab or space delimiters
            parts = line.split(maxsplit=1)
            if len(parts) != 2:
                continue  # why: ignore malformed lines quietly
            vid, text = parts[0], parts[1].strip()
            if vid in verses:
                # why: last one wins; avoids duplication noise if file has repeats
                pass
            verses[vid] = text

    return verses

# Example usage — set your file paths
eng_path = "eng-x-bible-newsimplified.txt"      # e.g., ENG file
other_path = "cmn-x-bible-newsimplified.txt"     # e.g., DEU file

eng_map = load_bible_txt_as_dict(eng_path)
other_map = load_bible_txt_as_dict(other_path)

print(f"ENG verses:  {len(eng_map)}")
print(f"OTHER verses:{len(other_map)}")
# Quick spot-check:
for k in list(eng_map.keys())[:5]:
    print(k, "→", eng_map[k][:60], "...")

ENG verses:  31035
OTHER verses:31074
01001001 → In the beginning God created the heavens and the earth . ...
01001002 → The earth was without form and empty . Darkness covered the  ...
01001003 → Then God said : » Let there be light ! « And there was light ...
01001004 → God saw that the light was good . God separated the light fr ...
01001005 → God called the light day . He called the darkness night . Th ...


In [8]:
import pandas as pd
from pathlib import Path
import csv

# === Step 5: Build bilingual CSV of matching perfect-like verses ===

# Input files
plg_path = Path("plg_verses.csv")  # from Step 3
eng_path = Path("eng-x-bible-newsimplified.txt")
other_path = Path("cmn-x-bible-newsimplified.txt")

# Language codes
lang1 = "ENG"
lang2 = "CMN"

# --- Load helper from previous cell (should already be defined) ---
# load_bible_txt_as_dict()

eng_map = load_bible_txt_as_dict(eng_path)
other_map = load_bible_txt_as_dict(other_path)

# --- Load list of verse IDs with perfect-like constructions ---
plg_ids = set(pd.read_csv(plg_path)["verse_id"].astype(str))

print(f"Loaded {len(plg_ids)} verse IDs from {plg_path.name}")

# --- Keep only verse IDs that appear in BOTH files ---
common_ids = [vid for vid in plg_ids if vid in eng_map and vid in other_map]
print(f"✅ {len(common_ids)} verses found in both ENG and {lang2}")

# --- Build DataFrame ---
records = []
for vid in common_ids:
    records.append({"verse_id": vid, "language": lang1, "text": eng_map[vid]})
    records.append({"verse_id": vid, "language": lang2, "text": other_map[vid]})

df_out = pd.DataFrame(records).sort_values(["verse_id", "language"]).reset_index(drop=True)

# --- Save output ---
output_path = Path(f"bible_{lang1.lower()}_{lang2.lower()}_perfects.csv")
df_out.to_csv(output_path, index=False, quoting=csv.QUOTE_ALL, encoding="utf-8")

print(f"✅ Done! Created bilingual CSV: {output_path}")
print(f"Rows: {len(df_out)}  (≈ {len(common_ids)} verses × 2)")
display(df_out.head(6))

Loaded 1953 verse IDs from plg_verses.csv
✅ 1499 verses found in both ENG and CMN
✅ Done! Created bilingual CSV: bible_eng_cmn_perfects.csv
Rows: 2998  (≈ 1499 verses × 2)


,verse_id,language,text
0,10002007,CMN,现在你们要坚强 ， 要作勇敢的人 ， 因为你们的主扫罗已经死了 ； 犹大家已经膏立了我作他们...
1,10002007,ENG,» Be strong and courageous . Your master Saul ...
2,10003008,CMN,押尼珥因伊施波设的话非常忿怒 ， 说 ： “我是犹大的狗头吗？今天我厚待你的父亲扫罗家、他的...
3,10003008,ENG,Ishbosheth’s question made Abner very angry . ...
4,10003019,CMN,押尼珥也把这话说给便雅悯人听 ， 然后又去希伯仑 ， 把以色列人和便雅悯全都同意的 一 切事...
5,10003019,ENG,Abner also spoke to the people of Benjamin . T...


In [9]:
import pandas as pd
from pathlib import Path
import csv

# === Step 6: Build multilingual CSV of matching PLG verses ===

# Input files
plg_path = Path("plg_verses.csv")  # the list of PLG verse IDs
bible_files = {
    "ENG": Path("eng_bib.txt"),
    "DEU": Path("deu_bib.txt"),
    "CMN": Path("cmn_bib.txt"),
    "JPN": Path("jpn_bib.txt"),
}

# --- Load all Bibles into dictionaries ---
bibles = {lang: load_bible_txt_as_dict(path) for lang, path in bible_files.items()}
for lang, verses in bibles.items():
    print(f"{lang}: {len(verses)} verses loaded")

# --- Load list of verse IDs with PLGs ---
plg_ids = set(pd.read_csv(plg_path)["verse_id"].astype(str))
print(f"PLG verse IDs: {len(plg_ids)}")

# --- Keep only verses present in *all four* files ---
common_ids = plg_ids.copy()
for lang, verses in bibles.items():
    common_ids &= set(verses.keys())

print(f"✅ {len(common_ids)} verses found in ALL {len(bibles)} languages")

# --- Build DataFrame ---
records = []
for vid in sorted(common_ids):
    for lang, verses in bibles.items():
        records.append({
            "verse_id": vid,
            "lang": lang,
            "text": verses[vid],
        })

df_out = pd.DataFrame(records)

# --- Save output ---
output_path = Path("bible_multilingual_plg_test.csv")
df_out.to_csv(output_path, index=False, quoting=csv.QUOTE_ALL, encoding="utf-8")

print(f"✅ Done! Created multilingual CSV: {output_path}")
print(f"Rows: {len(df_out)}  (≈ {len(common_ids)} verses × {len(bibles)})")
display(df_out.head(12))

ENG: 7927 verses loaded
DEU: 7957 verses loaded
CMN: 7940 verses loaded
JPN: 7960 verses loaded
PLG verse IDs: 1953
✅ 394 verses found in ALL 4 languages
✅ Done! Created multilingual CSV: bible_multilingual_plg_test.csv
Rows: 1576  (≈ 394 verses × 4)


,verse_id,lang,text
0,40003002,ENG,"» Repent , for the kingdom of heaven has come ..."
1,40003002,DEU,Tut Busse ! denn das Reich der Himmel ist gena...
2,40003002,CMN,说 ： “天国近了 ， 你们应当悔改。”
3,40003002,JPN,「悔い改めよ、天国は近づいた」。
4,40003017,ENG,Then a voice from heaven said : » This is my S...
5,40003017,DEU,"Und siehe , eine Stimme aus den Himmeln sprach..."
6,40003017,CMN,又有声音从天上来 ， 说 ： “这是我的爱子 ， 我所喜悦的。”
7,40003017,JPN,また天から声があって言った、「これはわたしの愛する子、わたしの心にかなう者である」。
8,40005027,ENG,"» You have heard it was said , ‘ You should no..."
9,40005027,DEU,"Ihr habt gehört , dass gesagt ist : "" Du solls..."


In [18]:
import spacy
nlp = spacy.load("en_core_web_sm")

def has_have_vbn(text: str) -> bool:
    doc = nlp(text)
    for i, tok in enumerate(doc):
        if tok.pos_ == "AUX" and tok.lemma_ == "have":
            for j in range(i + 1, len(doc)):
                if doc[j].pos_ in {"VERB", "AUX"} and doc[j].tag_ == "VBN":
                    return True
            return False
    return False

# New Testament IDs in your ENG Bible (books 40–66)
eng_nt_ids = [vid for vid in bibles["ENG"].keys() if int(vid) >= 40000000]

# Count verses that match the rule
eng_nt_have_vbn = sum(has_have_vbn(bibles["ENG"][vid]) for vid in eng_nt_ids)
eng_nt_have_vbn

890

In [10]:
[len(set(load_bible_txt_as_dict(p).keys())) for p in bible_files.values()]

[7927, 7957, 7940, 7960]

In [11]:
[len(plg_ids & set(v.keys())) for v in bibles.values()]

[394, 394, 394, 394]

In [13]:
import pandas as pd
df_plg = pd.read_csv("plg_verses.csv")
df_plg.head(), df_plg.columns.tolist(), len(df_plg)

(   verse_id                                       english_text
 0   1001031  god see all that -pron- have make . -pron- be ...
 1   1002003  god bless the seventh day and make -pron- holy...
 2   1003013  then jehovah god say to the eve : » what be th...
 3   1003022  jehovah say : » the man have now become like o...
 4   1006005  jehovah see how great man 's wickedness on the...,
 ['verse_id', 'english_text'],
 1953)

In [14]:
for lang, verses in bibles.items():
    overlap = len(set(df_plg["verse_id"].astype(str)) & set(verses.keys()))
    print(f"{lang}: {overlap}")

ENG: 394
DEU: 394
CMN: 394
JPN: 394


In [15]:
list(bibles["ENG"].keys())[:10]

['40001001',
 '40001002',
 '40001003',
 '40001004',
 '40001005',
 '40001006',
 '40001007',
 '40001008',
 '40001009',
 '40001010']

In [17]:
(df_plg["verse_id"].astype(int) >= 40000000).sum()

np.int64(394)